# Optimizacija hiperparametara — LSTM (Optuna)

U ovom notebooku pokrećemo **automatsku pretragu hiperparametara** za LSTM model koristeći [Optuna](https://optuna.org/) biblioteku.

**Search space:**

| Parametar | Opseg |
|-----------|-------|
| Learning rate | 1e-5 – 1e-2 (log scale) |
| Batch size | 16, 32, 64 |
| Dropout | 0.1 – 0.5 |
| Broj LSTM slojeva | 1, 2, 3 |

- **20 trial-a**, maksimalno **3 epohe** po trial-u (radi brzine)
- Optimizacija samo za **LSTM** (BERT je prespor za ovakvu pretragu)
- Ciljna metrika: **macro F1** na validation skupu

> **Napomena:** Puni HPO na CPU može trajati 1–2+ sata. Za brži test smanjite `n_trials`.

## 1. Priprema okruženja

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.hyperparameter_search import run_hyperparameter_search
from src.train import get_device

device = get_device()
print(f"Uređaj: {device}")
print(f"Broj trial-a: {config.HPO_N_TRIALS}")
print(f"Epohe po trial-u: {config.HPO_MAX_EPOCHS}")

## 2. Pokretanje pretrage hiperparametara

In [ ]:
study = run_hyperparameter_search(n_trials=config.HPO_N_TRIALS, device=device)

## 3. Pregled rezultata

In [ ]:
results_df = pd.read_csv(config.HPO_RESULTS_CSV)
value_col = "value" if "value" in results_df.columns else "values_0"
top_trials = results_df.sort_values(value_col, ascending=False).head(5)
top_trials

## 4. Najbolji hiperparametri

In [ ]:
print(f"Najbolji macro F1 (validation): {study.best_value:.4f}")
print("\nNajbolji hiperparametri:")
for param, value in study.best_params.items():
    print(f"  {param}: {value}")

## 5. Vizualizacije

Grafici su sačuvani u `results/hpo/` folderu tokom pretrage.

In [ ]:
hpo_dir = config.HPO_VISUALIZATIONS_DIR
for plot_name in ["param_importance", "parallel_coordinate", "optimization_history"]:
    png_path = hpo_dir / f"{plot_name}.png"
    html_path = hpo_dir / f"{plot_name}.html"
    if png_path.exists():
        print(f"\n{plot_name}:")
        display(Image(filename=str(png_path)))
    elif html_path.exists():
        print(f"{plot_name}: otvorite {html_path}")
    else:
        print(f"{plot_name}: grafikon nije dostupan")

## Zaključak

Pretraga hiperparametara je završena. Rezultati su sačuvani u:
- `results/hyperparameter_search.csv` — svi trial-i
- `results/hpo/` — vizualizacije (importance, parallel coordinate, optimization history)

Preporuka: koristite najbolje hiperparametre za finalni LSTM trening u `04_training.ipynb`.